# unbroadcast-pattern composite — cx30: unbroadcast inverts broadcast_to — sum out the axes that were expanded

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `unbroadcast-pattern`, `sum-and-broadcast-duality`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "unbroadcast-pattern"
DD_ATOM_IDS = ["unbroadcast-pattern", "sum-and-broadcast-duality"]
DD_SUBTOPICS = ["Backprop: Unbroadcast pattern", "Backprop: sum/broadcast duality"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

`np.broadcast_to(x, shape)` (forward) expands `x` into a larger shape by prepending 1-axes and/or stretching size-1 axes. `unbroadcast(grad, original)` (backward) does the exact inverse: sum over the prepended leading axes (they didn't exist in `original`), then sum out the size-1 axes (with `keepdim=True` so the shape matches). The two are exact duals — broadcasting in the forward pass is exactly summing in the backward pass.

This is the sum-and-broadcast-duality atom at its purest: same axes, opposite operations. The composite drill verifies that `unbroadcast(broadcast_to(x, big_shape), x).shape == x.shape` and that the gradient values are the count-correct sums.

### Composite Exercise — unbroadcast inverts broadcast_to — sum out the axes that were expanded

**Atoms exercised together**: `unbroadcast-pattern`, `sum-and-broadcast-duality`

Implement `cx30_unbroadcast_after_broadcast(x, big_shape)` that:

1. Computes `expanded = t.broadcast_to(x, big_shape).clone()` — the forward broadcast.
2. Calls an inline `unbroadcast(grad, original)` to collapse `t.ones(big_shape)` back to `x.shape`. Use the unbroadcast pattern: peel leading axes with `sum(dim=0)`; then collapse size-1 axes with `sum(dim=i, keepdim=True)`.
3. Returns `{'expanded_shape': expanded.shape, 'collapsed_grad': <unbroadcast result>}`.

The collapsed grad must have shape `x.shape` and values equal to the number of broadcast copies at each position — that's the count-of-summed-positions, which is the sum-broadcast duality at work.

In [ ]:
def cx30_unbroadcast_after_broadcast(x, big_shape):
    # Forward: expand x into big_shape (zero-copy view; clone for cleanliness).
    expanded = t.broadcast_to(x, big_shape).clone()

    # Backward dual: unbroadcast collapses the axes broadcast_to added/expanded.
    def unbroadcast(grad, original):
        # Step 1: peel leading axes that broadcasting prepended.
        while grad.ndim > original.ndim:
            grad = grad.sum(dim=0)
        # Step 2: collapse size-1 axes that were stretched, KEEPING shape.
        for i, size in enumerate(original.shape):
            if size == 1 and grad.shape[i] != 1:
                grad = grad.sum(dim=i, keepdim=True)
        return grad

    grad_in = t.ones(big_shape, dtype=x.dtype if x.is_floating_point() else t.float32)
    collapsed_grad = unbroadcast(grad_in, x)
    return {'expanded_shape': tuple(expanded.shape), 'collapsed_grad': collapsed_grad}


<details><summary>Show solution — cx30</summary>

```python
def cx30_unbroadcast_after_broadcast(x, big_shape):
    # Forward: expand x into big_shape (zero-copy view; clone for cleanliness).
    expanded = t.broadcast_to(x, big_shape).clone()

    # Backward dual: unbroadcast collapses the axes broadcast_to added/expanded.
    def unbroadcast(grad, original):
        # Step 1: peel leading axes that broadcasting prepended.
        while grad.ndim > original.ndim:
            grad = grad.sum(dim=0)
        # Step 2: collapse size-1 axes that were stretched, KEEPING shape.
        for i, size in enumerate(original.shape):
            if size == 1 and grad.shape[i] != 1:
                grad = grad.sum(dim=i, keepdim=True)
        return grad

    grad_in = t.ones(big_shape, dtype=x.dtype if x.is_floating_point() else t.float32)
    collapsed_grad = unbroadcast(grad_in, x)
    return {'expanded_shape': tuple(expanded.shape), 'collapsed_grad': collapsed_grad}
```

The sum-broadcast duality is the deepest invariant of this drill: forward `broadcast_to` EXPANDS, backward `unbroadcast` SUMS, and they cancel on size-1 axes. The values in `collapsed_grad` count how many broadcast copies summed into each original position — that's literally the size of the expanded axis. If you used `keepdim=False` in step 2 you'd drop the size-1 axes and the shape assert would fail.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx30'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx30',
        'subtopics': ["Backprop: Unbroadcast pattern", "Backprop: sum/broadcast duality"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()